In [ ]:
import numpy as np
import pandas as pd
import torch, os, pickle
from tqdm.autonotebook import tqdm

# what is our list of datasets?
datasets = [
    "acs", "alphafold", "bodmas", "hiv", 
    "llm_evals/mmlu-pro", "llm_evals/bbh+gpqa+ifeval+math+musr", "pew", "ulb"]

# directory
if "summaries" not in os.listdir():
    os.mkdir("summaries")

# 1. Pool-sizes for all datasets.

In [ ]:
# dictionary to store all of our dataset sizes
pool_sizes = {}

for dataset in datasets:
    
    # load in the dataset
    if "y_test.pt" in os.listdir(f"datasets/{dataset}"):
        y_test = torch.load(f"datasets/{dataset}/y_test.pt")
    else:
        y_test = torch.load(f"datasets/{dataset}/Y_test.pt")
        
    # llm-eval datasets have Y_test that are (N_models, N_questions)
    n_y = y_test.shape[0] if len(y_test.shape) == 1 else y_test.shape[1]
    
    # add to our list
    pool_sizes[dataset] = n_y
    
# save
with open("summaries/pool_sizes.pickle", "wb") as file:
    pickle.dump(pool_sizes, file)

# 2. ESS multipliers + coverage w.r.t. uniform with mean+SEM, per dataset+method.

In [ ]:
# columns to group by
GROUP_COLS = ["dataset", "budget", "method"]

In [ ]:
# slightly different processing routines per type of dataset (handle "alphafold" separately)
standard_datasets = ["acs", "bodmas", "hiv", "pew", "ulb"]
llm_evals_datasets = ["llm_evals/mmlu-pro", "llm_evals/bbh+gpqa+ifeval+math+musr"]

In [ ]:
# create a list of dataframes, one per dataset
summaries = []

# go thru each of our datasets
for dataset in tqdm(standard_datasets):
    
    # get the results for uniform and PPI
    baselines = pd.read_csv(f"logs/{dataset}_baselines.csv")
    uniform = baselines.query("method == 'uniform'").reset_index(drop=True)
    ppi = baselines.query("method == 'ppi'").reset_index(drop=True)
    
    # load in PAI and SAI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")
    sai = pd.read_csv(f"logs/{dataset}_sai.csv")
    
    # get the portion of uniform to merge with everybody else
    uniform_to_merge = uniform[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "uniform_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, uniform_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier"] = (pai.uniform_width / pai.width) ** 2
    sai = pd.merge(sai, uniform_to_merge, on=["dataset", "budget", "seed"])
    sai["ess_multiplier"] = (sai.uniform_width / sai.width) ** 2
    ppi = pd.merge(ppi, uniform_to_merge, on=["dataset", "budget", "seed"])
    ppi["ess_multiplier"] = (ppi.uniform_width / ppi.width) ** 2
    uniform["ess_multiplier"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    sai = sai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    sai.columns = ["_".join(c).strip("_") for c in sai.columns]
    ppi = ppi.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    ppi.columns = ["_".join(c).strip("_") for c in ppi.columns]
    uniform = uniform.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    uniform.columns = ["_".join(c).strip("_") for c in uniform.columns]
    
    # create our summary table
    summary = pd.concat([pai, sai, ppi, uniform]).reset_index(drop=True)
    summaries.append(summary)
    
# stitch everything together
summaries = pd.concat(summaries).reset_index(drop=True)

In [ ]:
# ALPHAFOLD-specific processing
uniform = pd.read_csv(f"logs/alphafold_uniform.csv")
ppi = pd.read_csv(f"logs/alphafold_ppi.csv")
pai = pd.read_csv(f"logs/alphafold_pai.csv")
sai = pd.read_csv(f"logs/alphafold_bai.csv").replace("bai", "sai") # just for consistency

# get the portion of uniform to merge with everybody else
uniform_to_merge = uniform[["dataset", "budget", "seed", "width"]]\
.rename(columns={"width" : "uniform_width"})

# merge + compute relative diagnostics
pai = pd.merge(pai, uniform_to_merge, on=["dataset", "budget", "seed"])
pai["ess_multiplier"] = (pai.uniform_width / pai.width) ** 2
sai = pd.merge(sai, uniform_to_merge, on=["dataset", "budget", "seed"])
sai["ess_multiplier"] = (sai.uniform_width / sai.width) ** 2
ppi = pd.merge(ppi, uniform_to_merge, on=["dataset", "budget", "seed"])
ppi["ess_multiplier"] = (ppi.uniform_width / ppi.width) ** 2
uniform["ess_multiplier"] = 1.0

# compute mean and sem + polish
pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
pai.columns = ["_".join(c).strip("_") for c in pai.columns]
sai = sai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
sai.columns = ["_".join(c).strip("_") for c in sai.columns]
ppi = ppi.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
ppi.columns = ["_".join(c).strip("_") for c in ppi.columns]
uniform = uniform.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
uniform.columns = ["_".join(c).strip("_") for c in uniform.columns]

# create our summary table
alphafold_summary = pd.concat([pai, sai, ppi, uniform]).reset_index(drop=True)
summaries = pd.concat([summaries, alphafold_summary]).reset_index(drop=True)

In [ ]:
# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # get the results for uniform and PPI
    baselines = pd.read_csv(f"logs/{dataset}_baselines.csv")
    baselines = baselines\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    uniform = baselines.query("method == 'uniform'").reset_index(drop=True)
    ppi = baselines.query("method == 'ppi'").reset_index(drop=True)
    
    # load in PAI and SAI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    sai = pd.read_csv(f"logs/{dataset}_sai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of uniform to merge with everybody else
    uniform_to_merge = uniform[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "uniform_width"})

    # merge + compute relative diagnostics
    pai = pd.merge(pai, uniform_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier"] = (pai.uniform_width / pai.width) ** 2
    sai = pd.merge(sai, uniform_to_merge, on=["dataset", "budget", "seed"])
    sai["ess_multiplier"] = (sai.uniform_width / sai.width) ** 2
    ppi = pd.merge(ppi, uniform_to_merge, on=["dataset", "budget", "seed"])
    ppi["ess_multiplier"] = (ppi.uniform_width / ppi.width) ** 2
    uniform["ess_multiplier"] = 1.0

    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    sai = sai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    sai.columns = ["_".join(c).strip("_") for c in sai.columns]
    ppi = ppi.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    ppi.columns = ["_".join(c).strip("_") for c in ppi.columns]
    uniform = uniform.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier"]].agg(["mean", "sem"]).reset_index()
    uniform.columns = ["_".join(c).strip("_") for c in uniform.columns]
    
    # create our summary table
    llm_evals_summary = pd.concat([pai, sai, ppi, uniform]).reset_index(drop=True)
    summaries = pd.concat([summaries, llm_evals_summary]).reset_index(drop=True)
    
# save everything
summaries.to_csv("summaries/summaries.csv", index=False)

# 3. LLM evaluations only: show gains against PAI (with-replacement).

In [ ]:
# list of dataframes for LLM-evals ablation only
llm_evals_ablation_summaries = []

# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # a. without-replacement PAI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # b. with-replacement PAI
    pai_wr = pd.read_csv(f"logs/{dataset}_pai_wr.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])\
    .replace("pai", "pai_wr")
    
    # c. quantities to merge + merge them
    pai_wr_to_merge = pai_wr[["dataset", "budget", "seed", "width"]].rename(columns={"width" : "pai_wr_width"})
    pai = pd.merge(pai, pai_wr_to_merge)
    
    # d. compute relative ESS multiplier
    pai["ess_multiplier_wrt_wr"] = (pai.pai_wr_width / pai.width) ** 2
    
    # e. aggregate by taking the mean
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_wr"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    
    # f. add to list
    llm_evals_ablation_summaries.append(pai)

# stitch together + save
llm_evals_ablation_summaries = pd.concat(llm_evals_ablation_summaries).reset_index(drop=True)
llm_evals_ablation_summaries.to_csv("summaries/llm_evals_ablation_summaries.csv", index=False)

# 4. ESS ratios of PAI vs. PPAT on all datasets. Also PAI ablations.

In [ ]:
# create a list of dataframes, one per dataset
ppat_comparison_summaries = []

# go thru each of our datasets (can include "alphafold" because no need for baselines)
for dataset in tqdm(standard_datasets + ["alphafold"]):
    
    # load in PAI and PPAT
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")
    ppat = pd.read_csv(f"logs/{dataset}_ppat.csv")
    
    # get the portion of PPAT to merge with PAI
    ppat_to_merge = ppat[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppat_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, ppat_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_ppat"] = (pai.ppat_width / pai.width) ** 2
    ppat["ess_multiplier_wrt_ppat"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    ppat = ppat.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    ppat.columns = ["_".join(c).strip("_") for c in ppat.columns]
    
    # create our summary table
    summary = pd.concat([pai, ppat]).reset_index(drop=True)
    ppat_comparison_summaries.append(summary)
    
# stitch everything together
ppat_comparison_summaries = pd.concat(ppat_comparison_summaries).reset_index(drop=True)

In [ ]:
# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and PPAT
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    ppat = pd.read_csv(f"logs/{dataset}_ppat.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of PPAT to merge with PAI
    ppat_to_merge = ppat[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppat_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, ppat_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_ppat"] = (pai.ppat_width / pai.width) ** 2
    ppat["ess_multiplier_wrt_ppat"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    ppat = ppat.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    ppat.columns = ["_".join(c).strip("_") for c in ppat.columns]
    
    # create our summary table
    llm_evals_summary = pd.concat([pai, ppat]).reset_index(drop=True)
    ppat_comparison_summaries = pd.concat([ppat_comparison_summaries, llm_evals_summary]).reset_index(drop=True)

In [ ]:
# PAI ablation experiments on {HIV, Pew, BODMAS, ACS}
for dataset in tqdm(["hiv", "pew", "bodmas", "acs"]):
    
    # load in PAI and PPAT
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv").replace("pai", "pai_frozen")
    ppat = pd.read_csv(f"logs/{dataset}_ppat.csv")
    
    # get the portion of PPAT to merge with PAI
    ppat_to_merge = ppat[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppat_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, ppat_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_ppat"] = (pai_frozen.ppat_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    ppat_comparison_summaries = pd.concat([ppat_comparison_summaries, pai_frozen])

In [ ]:
# PAI ablation experiments on the llm evaluation suites
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and PPAT
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]
            ).mean().reset_index().drop(columns=["model_idx"]).replace("pai", "pai_frozen")
    ppat = pd.read_csv(f"logs/{dataset}_ppat.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of PPAT to merge with PAI
    ppat_to_merge = ppat[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppat_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, ppat_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_ppat"] = (pai_frozen.ppat_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppat"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    ppat_comparison_summaries = pd.concat([ppat_comparison_summaries, pai_frozen])
    
# save to .csv
ppat_comparison_summaries.to_csv("summaries/ppat_comparison_summaries.csv", index=False)

# 5. ESS ratios of PAI vs. PPI on all datasets, including PAI ablations.

In [ ]:
# create a list of dataframes, one per dataset
ppi_comparison_summaries = []

# go thru each of our datasets (no need alphafold because only have fixed ML model)
for dataset in tqdm(standard_datasets + ["alphafold"]):
    
    # load in PAI and PPI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")
    if dataset != "alphafold":
        ppi = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'ppi'").reset_index()
    else:
        ppi = pd.read_csv(f"logs/{dataset}_ppi.csv")
    
    # get the portion of PPI to merge with PAI
    ppi_to_merge = ppi[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppi_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, ppi_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_ppi"] = (pai.ppi_width / pai.width) ** 2
    ppi["ess_multiplier_wrt_ppi"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    ppi = ppi.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    ppi.columns = ["_".join(c).strip("_") for c in ppi.columns]
    
    # create our summary table
    summary = pd.concat([pai, ppi]).reset_index(drop=True)
    ppi_comparison_summaries.append(summary)
    
# stitch everything together
ppi_comparison_summaries = pd.concat(ppi_comparison_summaries).reset_index(drop=True)

In [ ]:
# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and PPI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    ppi = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'ppi'").reset_index()\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of PPI to merge with PAI
    ppi_to_merge = ppi[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppi_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, ppi_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_ppi"] = (pai.ppi_width / pai.width) ** 2
    ppi["ess_multiplier_wrt_ppi"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    ppi = ppi.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    ppi.columns = ["_".join(c).strip("_") for c in ppi.columns]
    
    # create our summary table
    llm_evals_summary = pd.concat([pai, ppi]).reset_index(drop=True)
    ppi_comparison_summaries = pd.concat([ppi_comparison_summaries, llm_evals_summary]).reset_index(drop=True)

In [ ]:
# PAI ablation experiments on {HIV, Pew, BODMAS, ACS}
for dataset in tqdm(["hiv", "pew", "bodmas", "acs"]):
    
    # load in PAI and PPI
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv").replace("pai", "pai_frozen")
    ppi = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'ppi'")
    
    # get the portion of PPI to merge with PAI
    ppi_to_merge = ppi[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppi_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, ppi_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_ppi"] = (pai_frozen.ppi_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    ppi_comparison_summaries = pd.concat([ppi_comparison_summaries, pai_frozen])

In [ ]:
# PAI ablation experiments on the llm evaluation suites
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and PPI
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]
            ).mean().reset_index().drop(columns=["model_idx"]).replace("pai", "pai_frozen")
    ppi = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'ppi'")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of PPI to merge with PAI
    ppi_to_merge = ppi[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "ppi_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, ppi_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_ppi"] = (pai_frozen.ppi_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_ppi"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    ppi_comparison_summaries = pd.concat([ppi_comparison_summaries, pai_frozen])
    
# save to .csv
ppi_comparison_summaries.to_csv("summaries/ppi_comparison_summaries.csv", index=False)

# 6. ESS ratios of PAI vs. SAI on all datasets, no need ablations.

In [ ]:
# create a list of dataframes, one per dataset
sai_comparison_summaries = []

# go thru each of our datasets (need alphafold because only have fixed ML model)
for dataset in tqdm(standard_datasets + ["alphafold"]):
    
    # load in PAI and SAI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")
    if dataset != "alphafold":
        sai = pd.read_csv(f"logs/{dataset}_sai.csv")
    else:
        sai = pd.read_csv(f"logs/{dataset}_bai.csv").replace("bai", "sai") # for indexing purposes
    
    # get the portion of SAI to merge with PAI
    sai_to_merge = sai[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "sai_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, sai_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_sai"] = (pai.sai_width / pai.width) ** 2
    sai["ess_multiplier_wrt_sai"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_sai"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    sai = sai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_sai"]].agg(["mean", "sem"]).reset_index()
    sai.columns = ["_".join(c).strip("_") for c in sai.columns]
    
    # create our summary table
    summary = pd.concat([pai, sai]).reset_index(drop=True)
    sai_comparison_summaries.append(summary)
    
# stitch everything together
sai_comparison_summaries = pd.concat(sai_comparison_summaries).reset_index(drop=True)

In [ ]:
# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and SAI
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    sai = pd.read_csv(f"logs/{dataset}_sai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of SAI to merge with PAI
    sai_to_merge = sai[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "sai_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, sai_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_sai"] = (pai.sai_width / pai.width) ** 2
    sai["ess_multiplier_wrt_sai"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_sai"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    sai = sai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_sai"]].agg(["mean", "sem"]).reset_index()
    sai.columns = ["_".join(c).strip("_") for c in sai.columns]
    
    # create our summary table
    llm_evals_summary = pd.concat([pai, sai]).reset_index(drop=True)
    sai_comparison_summaries = pd.concat([sai_comparison_summaries, llm_evals_summary]).reset_index(drop=True)
    
# save to .csv
sai_comparison_summaries.to_csv("summaries/sai_comparison_summaries.csv", index=False)

# 7. ESS ratios of PAI vs. Uniform on all datasets, including PAI ablations.

In [ ]:
# create a list of dataframes, one per dataset
unif_comparison_summaries = []

# go thru each of our datasets (no need alphafold because only have fixed ML model)
for dataset in tqdm(standard_datasets):
    
    # load in PAI and Uniform
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")
    if dataset in ["alphafold", "ulb"]:
        continue
    else:
        unif = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'uniform'")

    # get the portion of Uniform to merge with PAI
    unif_to_merge = unif[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "unif_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, unif_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_unif"] = (pai.unif_width / pai.width) ** 2
    unif["ess_multiplier_wrt_unif"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    unif = unif.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    unif.columns = ["_".join(c).strip("_") for c in unif.columns]
    
    # create our summary table
    summary = pd.concat([pai, unif]).reset_index(drop=True)
    unif_comparison_summaries.append(summary)
    
# stitch everything together
unif_comparison_summaries = pd.concat(unif_comparison_summaries).reset_index(drop=True)

In [ ]:
# go thru each of our datasets (note: 100x seeds, but 2.2K models per seed)
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and Uniform
    pai = pd.read_csv(f"logs/{dataset}_pai.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    unif = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'uniform'").reset_index()\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of Uniform to merge with PAI
    unif_to_merge = unif[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "unif_width"})
    
    # merge + compute relative diagnostics
    pai = pd.merge(pai, unif_to_merge, on=["dataset", "budget", "seed"])
    pai["ess_multiplier_wrt_unif"] = (pai.unif_width / pai.width) ** 2
    unif["ess_multiplier_wrt_unif"] = 1.0
    
    # compute mean and sem + polish
    pai = pai.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    pai.columns = ["_".join(c).strip("_") for c in pai.columns]
    unif = unif.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    unif.columns = ["_".join(c).strip("_") for c in unif.columns]
    
    # create our summary table
    llm_evals_summary = pd.concat([pai, unif]).reset_index(drop=True)
    unif_comparison_summaries = pd.concat([unif_comparison_summaries, llm_evals_summary]).reset_index(drop=True)

In [ ]:
# PAI ablation experiments on {HIV, Pew, BODMAS, ACS}
for dataset in tqdm(["hiv", "pew", "bodmas", "acs"]):
    
    # load in PAI and Uniform
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv").replace("pai", "pai_frozen")
    unif = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'uniform'")
    
    # get the portion of Uniform to merge with PAI
    unif_to_merge = unif[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "unif_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, unif_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_unif"] = (pai_frozen.unif_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    unif_comparison_summaries = pd.concat([unif_comparison_summaries, pai_frozen])

In [ ]:
# PAI ablation experiments on the llm evaluation suites
for dataset in tqdm(llm_evals_datasets):
    
    # load in PAI and Uniform
    pai_frozen = pd.read_csv(f"logs/{dataset}_pai_ablation.csv")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]
            ).mean().reset_index().drop(columns=["model_idx"]).replace("pai", "pai_frozen")
    unif = pd.read_csv(f"logs/{dataset}_baselines.csv").query("method == 'uniform'")\
    .groupby(["dataset", "budget", "tau", "method", "seed"]).mean().reset_index().drop(columns=["model_idx"])
    
    # get the portion of Uniform to merge with PAI
    unif_to_merge = unif[["dataset", "budget", "seed", "width"]]\
    .rename(columns={"width" : "unif_width"})
    
    # merge + compute relative diagnostics
    pai_frozen = pd.merge(pai_frozen, unif_to_merge, on=["dataset", "budget", "seed"])
    pai_frozen["ess_multiplier_wrt_unif"] = (pai_frozen.unif_width / pai_frozen.width) ** 2
    
    # compute mean and sem + polish
    pai_frozen = pai_frozen.groupby(GROUP_COLS)[["coverage", "width", "ess_multiplier_wrt_unif"]].agg(["mean", "sem"]).reset_index()
    pai_frozen.columns = ["_".join(c).strip("_") for c in pai_frozen.columns]
    
    # update our table
    unif_comparison_summaries = pd.concat([unif_comparison_summaries, pai_frozen])
    
# save to .csv
unif_comparison_summaries.to_csv("summaries/unif_comparison_summaries.csv", index=False)